# 1.7 Integrate access, demand and supply into block and subzone tables

The last preparation step joins everything into the two tables the analysis stages read: one row
per residential HDB block (walk access + children + flat mix) and one row per subzone (children,
dwelling units, centres, access shares, densities, income). Two decisions are implemented here:

- **Demand age band 0–6** (prof-confirmed 2026-10-30, refined 2026-11-02): childcare demand is
  children below 7. From the 5-year SingStat bands, `child_0_6 = child_0_4 + share_5_6 × child_5_9`,
  where `share_5_6` is the subzone's share of ages 5–6 within 5–9 from the SingStat single-year table
  (0.30–0.53). Subzones with fewer than 100 residents aged 5–9 fall back to the approved flat 0.40
  (SingStat rounds cells to 10, so small cells give shares of 0 or 1); `child_0_6_flat40` carries the
  flat rule everywhere as a robustness column. National result: 176,409 children 0–6 (67.9% of 0–9).
- **Subzone access measures are weighted by children 0–6**, so a subzone's walk time is the walk
  time its children experience, not the mean over blocks.

Original script: `scripts/build_subzone_stats.py` (Minh).

**Inputs**
- `output/hdb_walk_access.csv` (1_07); `output/hdb_blocks.gpkg`, layers `hdb_blocks` (1_02) and
  `subzones` (1_01); `output/hdb_blocks_children_final.gpkg` (1_04);
  `output/childcare_centres_2026.gpkg`, layer `centres` (1_05).
- `data/SubzoneSingleYearAge_2026.csv` — SingStat single-year-of-age resident population by subzone.
- `output/median_household_income_planning_area/planning_area_median_household_income_range.csv` —
  planning-area median household income band (2_01, Daryl; read only. Folder renamed 2026-11-13).

**Outputs**
- `output/hdb_access_children.csv` — 10,769 blocks: access measures, children 0–4 / 5–9 / 0–9 / 0–6,
  flat mix, flags; read by the pressure-index and hypothesis notebooks.
- `output/subzone_stats.csv` and layer `subzone_stats` in `output/hdb_blocks.gpkg` — all 332 subzones
  (`has_hdb` flags the 159 with residential HDB): children by band, dwelling units, `pct_1to3room`,
  centres in polygon (`centres_n`, `_cc`, `_kn`, `_fullday`), centres per 1,000 children (0–9 legacy
  and `_0_6`), centre and child densities, child-weighted nearest walk time, shares of children within
  10 / 15 min of a centre, MRT walk time, `pa_median_income_sgd`; read by every ESDA notebook.

In [1]:
import sys
from pathlib import Path

import re

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

GPKG = OUT / "hdb_blocks.gpkg"
ACCESS = OUT / "hdb_walk_access.csv"
CHILDREN = OUT / "hdb_blocks_children_final.gpkg"
CENTRES = OUT / "childcare_centres_2026.gpkg"      # layer "centres"
SINGLE_YEAR = DATA / "SubzoneSingleYearAge_2026.csv"
INCOME = OUT / "median_household_income_planning_area" / "planning_area_median_household_income_range.csv"
BLOCK_CSV = OUT / "hdb_access_children.csv"
SUBZONE_CSV = OUT / "subzone_stats.csv"

FLAT_SHARE = 0.40        # prof-approved rule of thumb; national single-year share is 0.388
MIN_POP_5_9 = 100        # SingStat rounds cells to 10: below this the subzone share is noise -> fallback

## 1. Block table: access + children + the 0–6 demand band

The walk-access table is keyed by `OBJECTID`, the children layer by `join_key`; `hdb_blocks` bridges
the two. Every block must have a child count (the assertion), since the demand weights depend on it.

In [2]:
def share_5_6_by_subzone():
    """Share of residents aged 5-6 among ages 5-9, per subzone (SUBZONE_N), from the SingStat
    single-year table. Subzones with fewer than MIN_POP_5_9 residents aged 5-9 get NaN (caller falls
    back to FLAT_SHARE), because SingStat rounds to 10 and small cells give shares of 0 or 1."""
    a = pd.read_csv(SINGLE_YEAR)
    a["Pop"] = pd.to_numeric(a["Pop"], errors="coerce")
    a["Age"] = pd.to_numeric(a["Age"], errors="coerce")
    a = a[a["Age"].between(5, 9)]
    g = a.groupby(a["SZ"].str.upper().str.strip())["Pop"]
    pop59 = g.sum()
    pop56 = a[a["Age"] <= 6].groupby(a["SZ"].str.upper().str.strip())["Pop"].sum().reindex(pop59.index).fillna(0)
    return (pop56 / pop59.where(pop59 >= MIN_POP_5_9)).rename("share_5_6")


# 1. Access per block, bridged to the children layer through join_key.
acc = pd.read_csv(ACCESS)
blocks = pyogrio.read_dataframe(GPKG, layer="hdb_blocks", read_geometry=False,
                                columns=["OBJECTID", "join_key"])
kids = pyogrio.read_dataframe(CHILDREN, read_geometry=False)
kids = kids.rename(columns={"1- and 2-Room Flats*": "units_1_2room", "3-Room Flats": "units_3room",
                            "4-Room Flats": "units_4room", "5-Room and Executive Flats": "units_5room_exec"})
kcols = ["join_key", "child_0_4", "child_5_9", "child_pop_0_9",
         "units_1_2room", "units_3room", "units_4room", "units_5room_exec", "review_flag"]
blk = acc.merge(blocks, on="OBJECTID", how="left").merge(kids[kcols], on="join_key", how="left")
assert blk["child_pop_0_9"].notna().all(), "blocks without child counts"

# 2. Children 0-6 with the per-subzone share of 5-6, flat 0.40 where the subzone cell is too small.
share = share_5_6_by_subzone()
blk["share_5_6"] = blk["SUBZONE_N"].str.upper().map(share)
n_fallback = blk["share_5_6"].isna().sum()
blk["share_5_6"] = blk["share_5_6"].fillna(FLAT_SHARE)
blk["child_0_6"] = blk["child_0_4"] + blk["share_5_6"] * blk["child_5_9"]
blk["child_0_6_flat40"] = blk["child_0_4"] + FLAT_SHARE * blk["child_5_9"]
print(f"share_5_6: {blk['share_5_6'].min():.3f}-{blk['share_5_6'].max():.3f} across blocks; "
      f"{n_fallback} blocks fell back to {FLAT_SHARE}; "
      f"children 0-6 = {blk['child_0_6'].sum():,.0f} ({blk['child_0_6'].sum() / blk['child_pop_0_9'].sum():.1%} of 0-9); "
      f"flat-40 variant {blk['child_0_6_flat40'].sum():,.0f}")

# 3. Access flags used for the subzone shares, then write the block table.
blk["has_centre_10min"] = blk["centres_n_within_10min"] >= 1
blk["has_centre_15min"] = blk["centres_n_within_15min"] >= 1
blk["mrt_within_15min"] = blk["mrt_exits_nearest_min"] <= 15
blk.to_csv(BLOCK_CSV, index=False)
print(f"block table: {len(blk):,} rows -> {BLOCK_CSV.relative_to(ROOT)}")
print(f"children 0-4 {blk['child_0_4'].sum():,.0f}, 5-9 {blk['child_5_9'].sum():,.0f}, 0-9 {blk['child_pop_0_9'].sum():,.0f}")

share_5_6: 0.300-0.526 across blocks; 55 blocks fell back to 0.4; children 0-6 = 176,409 (67.9% of 0-9); flat-40 variant 177,548
block table: 10,769 rows -> output/hdb_access_children.csv
children 0-4 122,640, 5-9 137,270, 0-9 259,910


## 2. Centres inside each subzone

Point-in-polygon counts by service model; `centres_fullday` is everything but kindergartens (KN),
the supply definition used in the full-day sensitivity.

In [3]:
# 1. Centres in the subzones' CRS, counted per subzone by service model.
sz = pyogrio.read_dataframe(GPKG, layer="subzones")
centres = pyogrio.read_dataframe(CENTRES, layer="centres").to_crs(sz.crs)
cj = gpd.sjoin(centres[["centre_name", "service_model", "geometry"]], sz[["SUBZONE_C", "geometry"]],
               how="inner", predicate="within")
csum = cj.groupby("SUBZONE_C").agg(centres_n=("centre_name", "size"),
                                   centres_cc=("service_model", lambda s: (s == "CC").sum()),
                                   centres_kn=("service_model", lambda s: (s == "KN").sum()),
                                   centres_fullday=("service_model", lambda s: (s != "KN").sum()))
print(f"centres inside a subzone: {len(cj):,} of {len(centres):,}; subzones with a centre: {len(csum)}")
print(csum.sum().to_string())

centres inside a subzone: 1,796 of 1,796; subzones with a centre: 238
centres_n          1796
centres_cc         1467
centres_kn          183
centres_fullday    1613


## 3. Subzone aggregates, weighted by children 0–6

For each subzone with HDB blocks: children by band, dwelling units and the share of 1–3-room flats
(a socio-economic proxy), the child-weighted nearest walk time to a centre and to an MRT exit, the
child-weighted number of centres within 10 minutes, and the shares of children with a centre within
10 / 15 minutes and an MRT exit within 15.

In [4]:
def wmean(values, weights):
    """Weighted mean ignoring NaNs and non-positive weights; NaN when nothing is left."""
    v, w = np.asarray(values, float), np.asarray(weights, float)
    ok = ~np.isnan(v) & ~np.isnan(w) & (w > 0)
    return np.average(v[ok], weights=w[ok]) if ok.any() and w[ok].sum() > 0 else np.nan


rows = []
for code_, g in blk.groupby("SUBZONE_C"):
    w = g["child_0_6"].to_numpy(float)      # demand weight: children 0-6
    w09 = g["child_pop_0_9"].to_numpy(float)
    units = g["total_dwelling_units"].sum()
    small = g["units_1_2room"].sum() + g["units_3room"].sum()
    rows.append({
        "SUBZONE_C": code_, "n_blocks": len(g), "dwelling_units": int(units),
        "pct_1to3room": 100 * small / units if units else np.nan,
        "children_0_4": g["child_0_4"].sum(), "children_5_9": g["child_5_9"].sum(),
        "children_0_9": w09.sum(),
        "children_0_6": w.sum(), "children_0_6_flat40": g["child_0_6_flat40"].sum(),
        "share_5_6": g["share_5_6"].iloc[0],
        "walk_nearest_min_cw": wmean(g["centres_nearest_min"], w),
        "walk_nearest_min_mean": g["centres_nearest_min"].mean(),
        "centres_within_10min_cw": wmean(g["centres_n_within_10min"], w),
        "share_children_within_10min": w[g["has_centre_10min"].to_numpy()].sum() / w.sum() if w.sum() else np.nan,
        "share_children_within_15min": w[g["has_centre_15min"].to_numpy()].sum() / w.sum() if w.sum() else np.nan,
        "mrt_nearest_min_cw": wmean(g["mrt_exits_nearest_min"], w),
        "share_children_mrt_within_15min": w[g["mrt_within_15min"].to_numpy()].sum() / w.sum() if w.sum() else np.nan,
    })
hdb = pd.DataFrame(rows)
print(f"{len(hdb)} subzones with residential HDB; children 0-6 total {hdb['children_0_6'].sum():,.0f}")

159 subzones with residential HDB; children 0-6 total 176,409


## 4. All 332 subzones: supply ratios, densities, planning-area income

Subzones without HDB keep their centre counts (and zero children), so maps show the whole island.
`centres_per_1000_children` (0–9) is kept for comparability with earlier work; the demand-band
version is `centres_per_1000_children_0_6`. Income is the midpoint of the planning area's median
household income band from Daryl's table ("over" bands get +1,250).

In [5]:
def income_midpoint(band):
    """Midpoint in S$ of a band like '$7,000 - $7,999'; open 'over' bands get lower bound + 1,250."""
    nums = [int(x.replace(",", "")) for x in re.findall(r"\$([\d,]+)", str(band))]
    if not nums:
        return np.nan
    return nums[0] + 1250 if "over" in str(band).lower() else (nums[0] + nums[-1] + 1) / 2


# 1. Subzone polygons + HDB aggregates + centre counts; has_hdb flags the 159 with blocks.
stats = sz[["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N", "REGION_N", "area_km2", "geometry"]].merge(
    hdb, on="SUBZONE_C", how="left").merge(csum, on="SUBZONE_C", how="left")
stats["has_hdb"] = stats["n_blocks"].notna()
for c in ["centres_n", "centres_cc", "centres_kn", "centres_fullday"]:
    stats[c] = stats[c].fillna(0).astype(int)

# 2. Supply per child and densities.
stats["centres_per_1000_children"] = np.where(stats["children_0_9"] > 0,
                                              1000 * stats["centres_n"] / stats["children_0_9"], np.nan)
stats["centres_per_1000_children_0_6"] = np.where(stats["children_0_6"] > 0,
                                                  1000 * stats["centres_n"] / stats["children_0_6"], np.nan)
stats["centre_density_km2"] = stats["centres_n"] / stats["area_km2"]
stats["child_density_km2"] = stats["children_0_9"] / stats["area_km2"]
stats["child_density_0_6_km2"] = stats["children_0_6"] / stats["area_km2"]

# 3. Planning-area median household income (band midpoint).
inc = pd.read_csv(INCOME)
inc["PLN_AREA_N"] = inc["planning_area"].str.upper().str.strip()
inc["pa_median_income_sgd"] = inc["median_income_range"].map(income_midpoint)
stats = stats.merge(inc[["PLN_AREA_N", "pa_median_income_sgd"]], on="PLN_AREA_N", how="left")
miss_inc = stats.loc[stats["has_hdb"] & stats["pa_median_income_sgd"].isna(), "PLN_AREA_N"].unique()
print(f"planning areas with an income band: {inc['pa_median_income_sgd'].notna().sum()}; "
      f"HDB subzones without one: {len(miss_inc)}" + (f" {sorted(miss_inc)}" if len(miss_inc) else ""))

planning areas with an income band: 30; HDB subzones without one: 3 ['CHANGI', 'ROCHOR', 'TENGAH']


## 5. Write the subzone table and report the headline distribution

In [6]:
stats = gpd.GeoDataFrame(stats, geometry="geometry", crs=sz.crs)
stats.drop(columns="geometry").to_csv(SUBZONE_CSV, index=False)
stats.to_file(GPKG, layer="subzone_stats", driver="GPKG")
h = stats[stats["has_hdb"]]
print(f"subzone table: {len(stats)} subzones ({h.shape[0]} with HDB) -> {SUBZONE_CSV.relative_to(ROOT)} + gpkg layer subzone_stats")
print(f"national children 0-6 on HDB blocks: {h['children_0_6'].sum():,.0f}; 0-9: {h['children_0_9'].sum():,.0f}")
print("HDB subzones, child-weighted nearest walk (min): median {:.1f}, p90 {:.1f}, max {:.1f}".format(
    h["walk_nearest_min_cw"].median(), h["walk_nearest_min_cw"].quantile(.9), h["walk_nearest_min_cw"].max()))
print("share of children within 10 min of a centre: median {:.0%}, min {:.0%}".format(
    h["share_children_within_10min"].median(), h["share_children_within_10min"].min()))
print("centres per 1,000 children 0-6: median {:.1f}, min {:.1f}, max {:.1f}".format(
    h["centres_per_1000_children_0_6"].median(), h["centres_per_1000_children_0_6"].min(), h["centres_per_1000_children_0_6"].max()))
worst = h.sort_values("share_children_within_10min").head(8)
print("\nlowest share of children 0-6 within 10-min walk (HDB subzones):")
print(worst[["SUBZONE_N", "PLN_AREA_N", "children_0_6", "centres_n", "walk_nearest_min_cw",
             "share_children_within_10min"]].round(2).to_string(index=False))

subzone table: 332 subzones (159 with HDB) -> output/subzone_stats.csv + gpkg layer subzone_stats
national children 0-6 on HDB blocks: 176,409; 0-9: 259,910
HDB subzones, child-weighted nearest walk (min): median 3.2, p90 4.9, max 8.0
share of children within 10 min of a centre: median 100%, min 94%
centres per 1,000 children 0-6: median 9.5, min 0.0, max 125.1

lowest share of children 0-6 within 10-min walk (HDB subzones):
        SUBZONE_N  PLN_AREA_N  children_0_6  centres_n  walk_nearest_min_cw  share_children_within_10min
   WOODLANDS WEST   WOODLANDS       2205.79          8                 4.21                         0.94
      BEDOK SOUTH       BEDOK       1161.90         17                 4.26                         0.96
      BEDOK NORTH       BEDOK       2581.09         28                 4.22                         0.96
       ANAK BUKIT BUKIT TIMAH        275.03         15                 2.43                         0.98
      YISHUN WEST      YISHUN       1328.93   

## 6. Files written

In [7]:
print(f"{BLOCK_CSV.relative_to(ROOT)}: {len(pd.read_csv(BLOCK_CSV)):,} rows")
print(f"{SUBZONE_CSV.relative_to(ROOT)}: {len(pd.read_csv(SUBZONE_CSV)):,} rows")
print(f"{GPKG.relative_to(ROOT)} layer subzone_stats: {pyogrio.read_info(GPKG, layer='subzone_stats')['features']} features")

output/hdb_access_children.csv: 10,769 rows
output/subzone_stats.csv: 332 rows
output/hdb_blocks.gpkg layer subzone_stats: 332 features
